# Import and Process BRFSS Data

In [1]:
import pandas as pd
import numpy as np

## Curate the list of columns we're pulling in

In [3]:
# TARGET VARIABLE
target = ['_MICHD']  # Ever had CHD or MI

# TREATMENT VARIABLE (for uplift model)
treatment = ['_TOTINDA']  # Leisure-time physical activity in past 30 days

# FEATURES

# Demographics ---
demographic_cols = [
    '_SEX',         # Sex
    '_AGEG5YR',     # Age in 5-year bands
    '_RACE',        # Race/ethnicity
    'MARITAL',      # Marital status — proxy for social support
    'VETERAN3',     # Veteran status
    'PREGNANT',     # Pregnancy status — affects CV risk interpretation
]

# --- Socioeconomic / Social Determinants ---
socioeconomic_cols = [
    '_EDUCAG',      # Education level (binned)
    '_INCOMG1',     # Income level (binned)
    'SDHBILLS',     # Unable to pay bills — financial stress marker
    'SDHTRNSP',     # Transportation barriers — affects healthcare access
    'HOWSAFE1',     # Perceived neighborhood safety
]

# --- General Health Status ---
health_status_cols = [
    '_RFHLTH',      # Binary good/poor health
    'PHYSHLTH',     # Days physical health not good (past 30 days)
    'MENTHLTH',     # Days mental health not good (past 30 days)
    'POORHLTH',     # Days poor health limited usual activities
]

# --- Cardiovascular Risk Factors ---
# Note: CVDINFR4 and CVDCRHD4 are dropped since they compose _MICHD (target leakage)
cardiovascular_cols = [
    'CVDSTRK3',     # History of stroke — strong CV risk marker
    '_RFBMI5',      # Overweight/obese flag
    '_BMI5',        # Continuous BMI — more informative than binary flag
]

# --- Diabetes (strong MI risk factor) ---
diabetes_cols = [
    'DIABETE4',     # Ever told had diabetes
    'PREDIAB2',     # Pre-diabetes or borderline diabetes
    'DIABTYPE',     # Type 1 or Type 2
    'INSULIN1',     # Currently on insulin — severity proxy
    'FEETSORE',     # Foot sores — diabetes complication proxy
]

# --- Smoking (strong MI risk factor) ---
smoking_cols = [
    '_SMOKER3',     # 4-level smoking status (daily/someday/former/never)
    'USENOW3',      # Smokeless tobacco use
    '_CURECI3',     # Current e-cigarette user
    'LCSFIRST',     # Age started smoking — exposure duration proxy
    'LCSNUMCG',     # Average cigarettes per day — dose proxy
    '_LCSYSMK',     # Years smoked
    '_LCSYQTS',     # Years since quitting
    'STOPSMK2',     # Attempted to quit in past 12 months
]

# --- Alcohol ---
alcohol_cols = [
    'DRNKANY6',     # Any drinking in past 30 days
    'AVEDRNK4',     # Average drinks per drinking day
    '_RFBING6',     # Binge drinker flag
    '_RFDRHV9',     # Heavy drinker flag
    '_DRNKWK3',     # Total drinks per week
    'MAXDRNKS',     # Max drinks on a single occasion
]

# --- Comorbidities ---
comorbidity_cols = [
    'CHCCOPD3',     # COPD/emphysema/chronic bronchitis
    'CHCKDNY2',     # Kidney disease
    'HAVARTH4',     # Arthritis
    '_LTASTH1',     # Ever had asthma
    'ADDEPEV3',     # Depressive disorder — linked to CV outcomes
    'CHCSCNC1',     # Non-melanoma skin cancer history
    'CHCOCNC1',     # Melanoma or other cancer history
]

# --- Mental Health / Wellbeing ---
mental_health_cols = [
    'LSATISFY',     # Life satisfaction
    'SDLONELY',     # Loneliness — linked to CV outcomes
]

# --- Functional Limitations ---
functional_cols = [
    'DEAF',         # Serious hearing difficulty
    'BLIND',        # Serious vision difficulty
    'DECIDE',       # Difficulty concentrating or making decisions
    'DIFFWALK',     # Difficulty walking or climbing stairs — physical capacity proxy
    'DIFFDRES',     # Difficulty dressing or bathing
    'DIFFALON',     # Difficulty doing errands alone
]

# --- Healthcare Access ---
healthcare_access_cols = [
    '_HLTHPL2',     # Has health insurance
    'PERSDOC3',     # Has a personal doctor
    'MEDCOST1',     # Couldn't afford doctor in past 12 months
    'CHECKUP1',     # Time since last routine checkup
]

# --- Preventive Care ---
preventive_cols = [
    'FLUSHOT7',     # Flu shot — proxy for general health engagement
    'PNEUVAC4',     # Pneumonia vaccine — proxy for general health engagement
]

# --- Diet ---
diet_cols = [
    'SSBSUGR2',     # Sugary soda consumption — diet quality proxy
]

# --- Adverse Childhood Experiences ---
# These are distal risk factors but linked to adult CV outcomes via chronic stress
ace_cols = [
    'ACEDEPRS',     # Lived with someone depressed/mentally ill
    'ACEDRINK',     # Lived with problem drinker
    'ACEDRUGS',     # Lived with drug user
    'ACEPRISN',     # Lived with someone incarcerated
    'ACEDIVRC',     # Parents divorced/separated
    'ACEPUNCH',     # Witnessed domestic violence
    'ACEHURT1',     # Physically hurt by parent/adult
    'ACEADSAF',     # Had protective adult (positive buffer)
]

# --- Cognitive Decline (potential confounder) ---
cognitive_cols = [
    'CIMEMLO1',     # Worsening memory/thinking difficulties
    'CDSOCIA1',     # Cognitive difficulties affecting work/social life
]

# --- Body Measures ---
body_cols = [
    '_BMI5',        # Continuous BMI
    'HTIN4',        # Height in inches — may be needed for other computations
]

# --- Survey Geography (optional — use if modeling regional variation) ---
geography_cols = [
    'MSCODE',       # Metropolitan status (center city/suburban/non-MSA)
]

Load Data

In [4]:
curated_cols = target + treatment + demographic_cols + socioeconomic_cols + health_status_cols + cardiovascular_cols + diabetes_cols + smoking_cols + alcohol_cols + comorbidity_cols + mental_health_cols + functional_cols + healthcare_access_cols + preventive_cols + diet_cols + ace_cols + cognitive_cols + body_cols + geography_cols

In [6]:
import_path = '/work/BRFSS_Dataset_2024.csv'
df = pd.read_csv(import_path, usecols=curated_cols)

In [8]:
df

,PHYSHLTH,MENTHLTH,POORHLTH,PERSDOC3,MEDCOST1,CHECKUP1,CVDSTRK3,CHCSCNC1,CHCOCNC1,CHCCOPD3,...,_EDUCAG,_INCOMG1,_SMOKER3,_CURECI3,_LCSYSMK,_LCSYQTS,DRNKANY6,_RFBING6,_DRNKWK3,_RFDRHV9
0,2.0,88.0,88.0,2.0,2.0,1.0,2.0,1.0,2.0,2.0,...,2.0,9.0,4.0,1.0,NaN,NaN,2.0,1.0,5.397605e-79,1.0
1,88.0,88.0,NaN,1.0,2.0,1.0,2.0,2.0,2.0,2.0,...,4.0,7.0,3.0,1.0,2.0,53.0,2.0,1.0,5.397605e-79,1.0
2,30.0,88.0,1.0,3.0,1.0,4.0,2.0,2.0,2.0,2.0,...,3.0,9.0,1.0,1.0,45.0,NaN,1.0,2.0,1.400000e+03,1.0
3,88.0,88.0,NaN,1.0,2.0,1.0,2.0,1.0,2.0,2.0,...,4.0,4.0,4.0,1.0,NaN,NaN,2.0,1.0,5.397605e-79,1.0
4,88.0,88.0,NaN,1.0,2.0,1.0,2.0,2.0,2.0,2.0,...,3.0,2.0,4.0,1.0,NaN,NaN,2.0,1.0,5.397605e-79,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
457665,77.0,2.0,77.0,1.0,1.0,3.0,2.0,2.0,2.0,2.0,...,1.0,9.0,9.0,9.0,NaN,NaN,9.0,9.0,9.990000e+04,9.0
457666,5.0,1.0,88.0,3.0,2.0,2.0,2.0,2.0,2.0,2.0,...,1.0,4.0,1.0,1.0,50.0,NaN,1.0,9.0,3.500000e+03,2.0
457667,88.0,88.0,NaN,2.0,2.0,3.0,2.0,2.0,2.0,2.0,...,4.0,5.0,4.0,1.0,NaN,NaN,1.0,1.0,1.400000e+02,1.0
457668,88.0,88.0,NaN,2.0,2.0,2.0,2.0,2.0,2.0,2.0,...,3.0,6.0,3.0,9.0,NaN,NaN,9.0,9.0,9.990000e+04,9.0


### Survey values are not consistent nor coherent\. Manipulating the values here so that unknown or refused\-to\-answer values are nulled out, rather than signifying higher value\. Also recoded 'No' values to '0'\.

In [10]:
# ── Replacement mappings ───────────────────────────────────────────────────────

REPLACE_88_TO_0 = {
    'PHYSHLTH': [77, 99],
    'MENTHLTH': [77, 99],
    'POORHLTH': [77, 99],
    'AVEDRNK4': [77, 99],
    'MAXDRNKS': [77, 99],
}

REPLACE_888_TO_0 = {
    'LCSNUMCG': [777, 999],
}

REPLACE_888_TO_NAN = {
    'LCSFIRST': [777, 888, 999],    # 888 = never smoked regularly → NaN
    'SSBSUGR2': [777, 888, 999]
}

REPLACE_7_8_9_TO_NAN = [
    'PERSDOC3', 'MEDCOST1', 'CHECKUP1', 'CVDSTRK3', 'CHCSCNC1', 'CHCOCNC1',
    'CHCCOPD3', 'ADDEPEV3', 'CHCKDNY2', 'HAVARTH4', 'DIABETE4', 'VETERAN3',
    'PREGNANT', 'DEAF', 'BLIND', 'DECIDE', 'DIFFWALK', 'DIFFDRES', 'DIFFALON',
    'USENOW3', 'FLUSHOT7', 'PNEUVAC4', 'PREDIAB2', 'DIABTYPE', 'INSULIN1',
    'FEETSORE', 'CIMEMLO1', 'CDSOCIA1', 'ACEDEPRS', 'ACEDRINK', 'ACEDRUGS',
    'ACEPRISN', 'ACEDIVRC', 'ACEPUNCH', 'ACEHURT1', 'ACEADSAF',
    'LSATISFY', 'SDLONELY', 'SDHBILLS', 'SDHTRNSP', 'HOWSAFE1', 'STOPSMK2',
    '_RFHLTH', '_HLTHPL2', '_TOTINDA', '_LTASTH1', '_RFBMI5',
    '_EDUCAG', '_INCOMG1', '_SMOKER3', '_CURECI3',
    'DRNKANY6', '_RFBING6', '_RFDRHV9',
    'MARITAL',
]

REPLACE_TO_0 = {
    'PREDIAB2': [3],        # 3 = No -> 0
    'PERSDOC3': [3],        # 3 = No -> 0
    'USENOW3':  [3],        # 3 = Not at all -> 0
    'DIABETE4': [3, 4],     # 3 = No, 4 = Pre-diabetes -> 0
}

REPLACE_SPECIAL = {
    '_DRNKWK3': (None, [99900]),
    'DIABAGE4': (None, [98, 99]),
}

REPLACE_NO_TO_0 = ['MEDCOST1', 'CVDSTRK3', 'CHCSCNC1', 'CHCOCNC1', 'CHCCOPD3', 'ADDEPEV3', 'CHCKDNY2', 'HAVARTH4',
'VETERAN3', 'PREGNANT', 'DEAF', 'BLIND', 'DECIDE', 'DIFFWALK', 'DIFFDRES', 'DIFFALON', 'FLUSHOT7',
'PNEUVAC4', 'INSULIN1', 'FEETSORE', 'CIMEMLO1', 'CDSOCIA1', 'ACEDEPRS', 'ACEDRINK', 'ACEDRUGS', 'ACEPRISN',
'ACEDIVRC', 'SDHBILLS', 'SDHTRNSP', 'STOPSMK2', '_HLTHPL2', '_TOTINDA', '_MICHD', '_LTASTH1',
'_RFBMI5', '_CURECI3', 'DRNKANY6', '_RFBING6', '_RFDRHV9'
]

# ── Helper function ────────────────────────────────────────────────────────────

def clean_col(series, zero_vals=None, nan_vals=None):
    if zero_vals is not None:
        for v in zero_vals:
            series = series.replace(v, 0)
    if nan_vals is not None:
        series = series.replace(nan_vals, np.nan)
    return series

# ── Apply replacements ─────────────────────────────────────────────────────────

for col, nan_vals in REPLACE_88_TO_0.items():
    if col in df.columns:
        df[col] = clean_col(df[col], zero_vals=[88], nan_vals=nan_vals)

for col, nan_vals in REPLACE_888_TO_0.items():
    if col in df.columns:
        df[col] = clean_col(df[col], zero_vals=[888], nan_vals=nan_vals)

for col, nan_vals in REPLACE_888_TO_NAN.items():
    if col in df.columns:
        df[col] = clean_col(df[col], nan_vals=nan_vals)

for col in REPLACE_7_8_9_TO_NAN:
    if col in df.columns:
        df[col] = clean_col(df[col], nan_vals=[7, 8, 9])

for col, zero_vals in REPLACE_TO_0.items():
    if col in df.columns:
        df[col] = clean_col(df[col], zero_vals=zero_vals)

for col, (zero_vals, nan_vals) in REPLACE_SPECIAL.items():
    if col in df.columns:
        df[col] = clean_col(df[col], zero_vals=zero_vals, nan_vals=nan_vals)

for col, (zero_vals, nan_vals) in REPLACE_SPECIAL.items():
    if col in df.columns:
        df[col] = clean_col(df[col], zero_vals=zero_vals, nan_vals=nan_vals)

for col in REPLACE_NO_TO_0:
    if col in df.columns:
        df[col] = df[col].replace(2, 0)

# ── _LCSYQTS: negative values are data errors → NaN ───────────────────────────

if '_LCSYQTS' in df.columns:
    df['_LCSYQTS'] = df['_LCSYQTS'].where(df['_LCSYQTS'] >= 0, np.nan)

# ── _BMI5: 2 implied decimal places ───────────────────────────────────────────

if '_BMI5' in df.columns:
    df['_BMI5'] = df['_BMI5'] / 100

# ── MSCODE: 5 → 0 (non-MSA as reference category) ────────────────────────────

if 'MSCODE' in df.columns:
    df['MSCODE'] = df['MSCODE'].replace(5, 0)

# Recode target and treatment variables from 1/2 to 1/0
df['_MICHD'] = df['_MICHD'].replace(2, 0)
df['_TOTINDA'] = df['_TOTINDA'].replace(2, 0)

In [12]:
df

,PHYSHLTH,MENTHLTH,POORHLTH,PERSDOC3,MEDCOST1,CHECKUP1,CVDSTRK3,CHCSCNC1,CHCOCNC1,CHCCOPD3,...,_EDUCAG,_INCOMG1,_SMOKER3,_CURECI3,_LCSYSMK,_LCSYQTS,DRNKANY6,_RFBING6,_DRNKWK3,_RFDRHV9
0,2.0,0.0,0.0,2.0,0.0,1.0,0.0,1.0,0.0,0.0,...,2.0,NaN,4.0,1.0,NaN,NaN,0.0,1.0,5.397605e-79,1.0
1,0.0,0.0,NaN,1.0,0.0,1.0,0.0,0.0,0.0,0.0,...,4.0,NaN,3.0,1.0,2.0,53.0,0.0,1.0,5.397605e-79,1.0
2,30.0,0.0,1.0,0.0,1.0,4.0,0.0,0.0,0.0,0.0,...,3.0,NaN,1.0,1.0,45.0,NaN,1.0,0.0,1.400000e+03,1.0
3,0.0,0.0,NaN,1.0,0.0,1.0,0.0,1.0,0.0,0.0,...,4.0,4.0,4.0,1.0,NaN,NaN,0.0,1.0,5.397605e-79,1.0
4,0.0,0.0,NaN,1.0,0.0,1.0,0.0,0.0,0.0,0.0,...,3.0,2.0,4.0,1.0,NaN,NaN,0.0,1.0,5.397605e-79,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
457665,NaN,2.0,NaN,1.0,1.0,3.0,0.0,0.0,0.0,0.0,...,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
457666,5.0,1.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,...,1.0,4.0,1.0,1.0,50.0,NaN,1.0,NaN,3.500000e+03,0.0
457667,0.0,0.0,NaN,2.0,0.0,3.0,0.0,0.0,0.0,0.0,...,4.0,5.0,4.0,1.0,NaN,NaN,1.0,1.0,1.400000e+02,1.0
457668,0.0,0.0,NaN,2.0,0.0,2.0,0.0,0.0,0.0,0.0,...,3.0,6.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
output_path = '/work/processed_data.csv'
df.to_csv(output_path, index=False)

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=88479c56-07bc-4e04-92b4-d8345dc226f4' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>